# První analýza dat – Ophir Vega PD300-IR (`data_correctedTimeStamp`)

Notebook načítá data s opravenou časovou osou (15 Hz, $\Delta t = 1/15\text{ s}$) z adresáře `data_correctedTimeStamp/`:
1. **Analýza jednotlivých souborů** – výpočet absolutního rozdílu dvou po sobě jdoucích hodnot $|\Delta y_i| = |y_{i+1} - y_i|$ a jednoduché numerické derivace $|\Delta y_i / \Delta t|$ (interaktivní prohlížení po jednotlivých teplotách + souhrnná tabulka).
2. **Porovnání všech 49 souborů v jednom grafu** – barevné rozlišení podle teploty (0 °C – 48 °C, střed 24 °C).

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Najdi adresář data_correctedTimeStamp (funguje při spuštění z notebooks/ i z kořene projektu)
DATA_DIR = Path("..") / "data_correctedTimeStamp"
if not DATA_DIR.is_dir():
    DATA_DIR = Path("data_correctedTimeStamp")
assert DATA_DIR.is_dir(), f"Adresář nenalezen: {DATA_DIR.resolve()}"


def extract_temp(path: Path) -> int:
    m = re.search(r"(-?\d+)", path.stem)
    return int(m.group(1)) if m else 9999


def load_starlab_file(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Načte sloupce Timestamp [s] a Channel A [W] z opraveného StarLab souboru."""
    t_vals, y_vals = [], []
    in_data = False
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not in_data:
                if "Timestamp" in line and "Channel A" in line:
                    in_data = True
                continue
            parts = line.strip().split()
            if len(parts) >= 2:
                t_vals.append(float(parts[0]))
                y_vals.append(float(parts[1]))
    return np.asarray(t_vals, dtype=np.float64), np.asarray(y_vals, dtype=np.float64)

# Načtení všech souborů seřazených numericky podle teploty
files = sorted(DATA_DIR.glob("*.txt"), key=extract_temp)
datasets = {}

for p in files:
    temp = extract_temp(p)
    t, y = load_starlab_file(p)
    dt = np.diff(t)
    dy_abs = np.abs(np.diff(y))          # |y[i+1] - y[i]|  [W]
    dydt_abs = dy_abs / dt               # |dy / dt|        [W/s]
    datasets[temp] = {
        "name": p.name,
        "t": t,
        "y": y,
        "t_diff": t[1:],                 # čas odpovídající rozdílu y[i+1] - y[i]
        "dy_abs": dy_abs,
        "dydt_abs": dydt_abs,
    }

temps = sorted(datasets.keys())
print(f"Načteno {len(datasets)} souborů z: {DATA_DIR.resolve()}")
print(f"Teploty: {temps[0]} °C až {temps[-1]} °C, dt = {np.median(np.diff(datasets[temps[0]]['t'])):.6f} s")

## 1. Jednotlivé soubory: $|\Delta y|$ a jednoduchá derivace $|\Delta y / \Delta t|$

Nejprve rychlá souhrnná tabulka pro všechny soubory a poté vykreslení vybraného souboru (lze měnit interaktivně pomocí výběru teploty).

In [2]:
rows = []
for temp in temps:
    d = datasets[temp]
    idx_max = int(np.argmax(d["dy_abs"]))
    rows.append({
        "Teplota [°C]": temp,
        "Soubor": d["name"],
        "Vzorků": len(d["t"]),
        "y_min [µW]": d["y"].min() * 1e6,
        "y_max [µW]": d["y"].max() * 1e6,
        "max |Δy| [µW]": d["dy_abs"][idx_max] * 1e6,
        "max |Δy/Δt| [µW/s]": d["dydt_abs"][idx_max] * 1e6,
        "t(max |Δy|) [s]": d["t_diff"][idx_max],
    })

summary_df = pd.DataFrame(rows)
summary_df.head(10)

NameError: name 'temps' is not defined

In [ ]:
def plot_single_temperature(temp: int, t_min: float = 0.0, t_max: float = 900.0) -> None:
    d = datasets[temp]
    mask_y = (d["t"] >= t_min) & (d["t"] <= t_max)
    mask_d = (d["t_diff"] >= t_min) & (d["t_diff"] <= t_max)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)

    # Horní graf: původní signál
    ax1.plot(d["t"][mask_y], d["y"][mask_y] * 1e6, color="tab:blue", lw=1.0)
    ax1.set_ylabel("Výkon Channel A [µW]")
    ax1.set_title(f"Soubor: {d['name']} ({temp} °C) – signál a absolutní rozdíl / derivace")

    # Dolní graf: |Δy| [µW] + sekundární osa pro |Δy/Δt| [µW/s]
    ax2.plot(d["t_diff"][mask_d], d["dy_abs"][mask_d] * 1e6, color="tab:red", lw=0.9)
    ax2.set_xlabel("Čas t [s]")
    ax2.set_ylabel("|Δy| [µW]", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax2_r = ax2.twinx()
    y1, y2 = ax2.get_ylim()
    ax2_r.set_ylim(y1 * 15.0, y2 * 15.0)
    ax2_r.set_ylabel("|Δy / Δt| [µW/s]  (fs = 15 Hz)", color="darkred")
    ax2_r.tick_params(axis="y", labelcolor="darkred")
    ax2_r.grid(False)

    plt.tight_layout()
    plt.show()


# Interaktivní výběr teploty a časového okna (při běžícím Jupyteru)
try:
    import ipywidgets as widgets
    from IPython.display import display

    w_temp = widgets.SelectionSlider(options=temps, value=temps[0], description="Teplota °C:")
    w_range = widgets.FloatRangeSlider(
        value=[0.0, 900.0], min=0.0, max=900.0, step=10.0, description="Čas [s]:", layout=widgets.Layout(width="500px")
    )
    out = widgets.interactive_output(
        lambda temp, tr: plot_single_temperature(temp, tr[0], tr[1]),
        {"temp": w_temp, "tr": w_range},
    )
    display(widgets.VBox([w_temp, w_range, out]))
except Exception:
    plot_single_temperature(temps[0])


# --- Průměrná hodnota signálu vs teplota ---

temps_plot = []
mean_y = []

for temp in temps:
    d = datasets[temp]

    temps_plot.append(temp)
    mean_y.append(np.mean(d["y"]) * 1e6)

plt.figure(figsize=(10, 5))

plt.plot(
    temps_plot,
    mean_y,
    marker="o"
)

plt.axvline(24, color="black", ls="--", alpha=0.5)

plt.xlabel("Teplota [°C]")
plt.ylabel("Průměrný výkon [µW]")
plt.title("Průměrná hodnota signálu vs teplota")

plt.grid(True, alpha=0.3)
plt.show()

## 2. Všechny soubory v jednom grafu (0 °C – 48 °C)

Barevná škála `coolwarm` zvýrazňuje teplotu vůči referenčním **24 °C** (modrá = nižší teploty, červená = vyšší teploty).
- **Graf 2A:** Absolutní rozdíl $|\Delta y(t)|$ (a pravá osa $|\Delta y / \Delta t|$) pro všech 49 souborů v celém čase 0–900 s.
- **Graf 2B:** Detail oblasti nástupu a kmitání (100–350 s) pro lepší čitelnost dynamiky.
- **Graf 2C:** Pro srovnání všechny původní průběhy signálu `Channel A` $y(t)$ v jednom grafu.

In [ ]:
norm = mcolors.TwoSlopeNorm(vmin=min(temps), vcenter=24, vmax=max(temps))
cmap = cm.coolwarm
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Nastavení rozsahu os (lze snadno upravit)
T_MIN, T_MAX = 140, 240   # časové okno [s]
Y_MAX_UW = 2.1            # rozsah osy Y: [-Y_MAX_UW, +Y_MAX_UW] µW

# --- Graf 2A: Všechny průběhy ±|Δy(t)| v okně 140–240 s (< 24 °C kladné, >= 24 °C záporné) ---
fig, ax = plt.subplots(figsize=(14, 7.0), constrained_layout=True)
for temp in temps:
    d = datasets[temp]
    sign = -1.0 if temp >= 24 else 1.0
    mask = (d["t_diff"] >= T_MIN) & (d["t_diff"] <= T_MAX)
    ax.plot(d["t_diff"][mask], sign * d["dy_abs"][mask] * 1e6, color=cmap(norm(temp)), lw=0.85, alpha=0.7)

ax.axhline(0, color="black", lw=0.8, alpha=0.6)
ax.set_title(f"Všechny soubory ({T_MIN}–{T_MAX} s): +|Δy| pro T < 24 °C,  -|Δy| pro T ≥ 24 °C")
ax.set_xlabel("Čas t [s]")
ax.set_ylabel("±|Δy| [µW]   (< 24 °C kladné, ≥ 24 °C záporné)")
ax.set_xlim(T_MIN, T_MAX)
ax.set_ylim(-Y_MAX_UW, Y_MAX_UW)

ax_r = ax.twinx()
y1, y2 = ax.get_ylim()
ax_r.set_ylim(y1 * 15.0, y2 * 15.0)
ax_r.set_ylabel("±|Δy / Δt| [µW/s]")
ax_r.grid(False)

cbar = fig.colorbar(sm, ax=ax_r, pad=0.07)
cbar.set_label("Teplota [°C]")
plt.show()

# --- Graf 2B: Celý průběh 0–900 s (pro kontext) ---
fig, ax = plt.subplots(figsize=(14, 5.5), constrained_layout=True)

for temp in temps:
    d = datasets[temp]

    sign = -1 if temp >= 24 else 1

    cumulative = np.cumsum(d["dy_abs"])

    ax.plot(
        d["t_diff"],
        sign * cumulative * 1e6,
        color=cmap(norm(temp)),
        lw=0.8,
        alpha=0.7
    )

ax.axhline(0, color="black", lw=0.8)

ax.set_title("Kumulativní Σ|Δy|")
ax.set_xlabel("Čas [s]")
ax.set_ylabel("±Σ|Δy| [µW]")
ax.set_xlim(140, 300)
ax.relim()
ax.autoscale()

plt.show()

# --- Graf 2BB: Derivace kumulativního Σ|Δy| ---

fig, ax = plt.subplots(figsize=(14, 5.5), constrained_layout=True)

for temp in temps:
    d = datasets[temp]

    sign = -1 if temp >= 24 else 1

    cumulative = np.cumsum(d["dy_abs"])

    # derivace podle času
    speed = np.gradient(cumulative, d["t_diff"])

    # vyhlazení (cca 5 s okno při 10 Hz)
    speed_smooth = (
        pd.Series(speed)
        .rolling(50, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )

    ax.plot(
        d["t_diff"],
        sign * speed_smooth * 1e6,
        color=cmap(norm(temp)),
        lw=0.8,
        alpha=0.7
    )

ax.axhline(0, color="black", lw=0.8)

ax.set_title("Graf 2BB: d(Σ|Δy|)/dt")
ax.set_xlabel("Čas [s]")
ax.set_ylabel("± d(Σ|Δy|)/dt [µW/s]")

ax.set_xlim(0, 900)

ax_r = ax.twinx()
y1, y2 = ax.get_ylim()
ax_r.set_ylim(y1 * 15.0, y2 * 15.0)
ax_r.set_ylabel("Alternativní měřítko")
ax_r.grid(False)

cbar = fig.colorbar(sm, ax=ax_r, pad=0.07)
cbar.set_label("Teplota [°C]")

plt.show()

# --- Graf 2BC: Konečná hodnota Σ|Δy| jako funkce teploty ---

temps_plot = []
final_values = []

for temp in temps:
    d = datasets[temp]

    cumulative = np.cumsum(d["dy_abs"])

    temps_plot.append(temp)
    final_values.append(cumulative[-1] * 1e6)

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax.plot(
    temps_plot,
    final_values,
    marker="o",
    lw=2
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title("Graf 2BC: Finální hodnota Σ|Δy|")
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("Σ|Δy| [µW]")

plt.show()

# --- Graf 2BD: časové těžiště změn tc(T) ---

temps_plot = []
tc_values = []

for temp in temps:
    d = datasets[temp]

    weights = d["dy_abs"]
    times = d["t_diff"]

    tc = np.sum(times * weights) / np.sum(weights)

    temps_plot.append(temp)
    tc_values.append(tc)

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax.plot(
    temps_plot,
    tc_values,
    marker="o",
    lw=2
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title("Graf 2BD: Časové těžiště změn")
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("tc [s]")

ax.grid(True, alpha=0.3)

plt.show()

# --- Graf 2BE: Kombinovaná metrika F(T) = Σ|Δy| / tc ---

temps_plot = []
feature_values = []

for temp in temps:
    d = datasets[temp]

    weights = d["dy_abs"]
    times = d["t_diff"]

    final_value = np.sum(weights)      # = Σ|Δy|
    tc = np.sum(times * weights) / np.sum(weights)

    feature = final_value / tc

    temps_plot.append(temp)
    feature_values.append(feature * 1e6)

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax.plot(
    temps_plot,
    feature_values,
    marker="o",
    lw=2
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title("Graf 2BE: Σ|Δy| / tc")
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("Σ|Δy| / tc [µW/s]")

ax.grid(True, alpha=0.3)

plt.show()

# --- Graf 2BF: Porovnání kandidátních metrik pro odhad teploty ---

temps_plot = []

sum_abs_list = []      # 2BC
tc_list = []           # těžiště
t50_list = []
t90_list = []
feature_list = []      # Σ|Δy| / tc

for temp in temps:
    d = datasets[temp]

    weights = d["dy_abs"]
    times = d["t_diff"]

    cum = np.cumsum(weights)

    final_value = cum[-1]

    tc = np.sum(times * weights) / np.sum(weights)

    idx50 = np.argmax(cum >= 0.50 * final_value)
    idx90 = np.argmax(cum >= 0.90 * final_value)

    t50 = times[idx50]
    t90 = times[idx90]

    feature = final_value / tc

    temps_plot.append(temp)

    sum_abs_list.append(final_value)
    tc_list.append(tc)
    t50_list.append(t50)
    t90_list.append(t90)
    feature_list.append(feature)


def normalize(x):
    x = np.asarray(x, dtype=float)
    xmin = np.min(x)
    xmax = np.max(x)

    if xmax == xmin:
        return np.zeros_like(x)

    return (x - xmin) / (xmax - xmin)


fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)

ax.plot(
    temps_plot,
    normalize(sum_abs_list),
    marker="o",
    lw=2,
    label="Σ|Δy|"
)

ax.plot(
    temps_plot,
    normalize(tc_list),
    marker="s",
    lw=2,
    label="tc"
)

ax.plot(
    temps_plot,
    normalize(t50_list),
    marker="^",
    lw=2,
    label="T50"
)

ax.plot(
    temps_plot,
    normalize(t90_list),
    marker="v",
    lw=2,
    label="T90"
)

ax.plot(
    temps_plot,
    normalize(feature_list),
    marker="D",
    lw=2,
    label="Σ|Δy| / tc"
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title("Graf 2BF: Porovnání metrik pro predikci teploty")
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("Normalizovaná hodnota [0–1]")

ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

# --- Graf 2BG: Integrál Σ|Δy| pouze v okně 140–400 s ---

temps_plot = []
window_energy = []

T_START = 140
T_STOP = 400

for temp in temps:
    d = datasets[temp]

    mask = (
        (d["t_diff"] >= T_START)
        & (d["t_diff"] <= T_STOP)
    )

    # fyzikálně správněji než prostý součet
    feature = np.trapz(
        d["dy_abs"][mask],
        d["t_diff"][mask]
    )

    temps_plot.append(temp)
    window_energy.append(feature * 1e6)

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax.plot(
    temps_plot,
    window_energy,
    marker="o",
    lw=2
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title(
    f"Graf 2BG: ∫|Δy|dt v okně {T_START}-{T_STOP} s"
)
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("∫|Δy|dt [µW·s]")

ax.grid(True, alpha=0.3)

plt.show()

# --- Graf 2BH: Integrál nad šumovým prahem v okně 140–400 s ---

temps_plot = []
feature_values = []

T_START = 140
T_STOP = 400
BASELINE_END = 100

for temp in temps:
    d = datasets[temp]

    # odhad šumu před dějem
    baseline_mask = d["t_diff"] < BASELINE_END
    noise_level = np.median(d["dy_abs"][baseline_mask])

    # odečítání šumového pozadí
    signal = np.clip(d["dy_abs"] - noise_level, 0, None)

    # vlastní přechodový děj
    mask = (
        (d["t_diff"] >= T_START)
        & (d["t_diff"] <= T_STOP)
    )

    # integrál pouze nad šumovým prahem
    feature = np.trapz(
        signal[mask],
        d["t_diff"][mask]
    )

    temps_plot.append(temp)
    feature_values.append(feature * 1e6)

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

ax.plot(
    temps_plot,
    feature_values,
    marker="o",
    lw=2
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title(
    f"Graf 2BH: ∫ max(|Δy|-šum,0) dt  ({T_START}-{T_STOP} s)"
)

ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("Integrovaná odezva [µW·s]")

ax.grid(True, alpha=0.3)

plt.show()

# --- Graf 2BI: 5°C binning pro 2BH ---

temps_plot = []
feature_values = []

T_START = 140
T_STOP = 400
BASELINE_END = 100

for temp in temps:
    d = datasets[temp]

    baseline_mask = d["t_diff"] < BASELINE_END
    noise_level = np.median(d["dy_abs"][baseline_mask])

    signal = np.clip(d["dy_abs"] - noise_level, 0, None)

    mask = (
        (d["t_diff"] >= T_START)
        & (d["t_diff"] <= T_STOP)
    )

    feature = np.trapz(
        signal[mask],
        d["t_diff"][mask]
    )

    temps_plot.append(temp)
    feature_values.append(feature * 1e6)

# -----------------------------
# binning po 5°C
# -----------------------------

df = pd.DataFrame({
    "temp": temps_plot,
    "feature": feature_values
})

df["bin"] = (df["temp"] // 5) * 5

stats = (
    df.groupby("bin")["feature"]
      .agg(["mean", "std", "count"])
      .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)

# původní body
ax.scatter(
    df["temp"],
    df["feature"],
    color="lightgray",
    s=40,
    label="Jednotlivá měření"
)

# průměry po 5°C
ax.errorbar(
    stats["bin"] + 2.5,
    stats["mean"],
    yerr=stats["std"],
    fmt="o-",
    capsize=5,
    lw=2,
    color="tab:blue",
    label="Průměr ± SD"
)

ax.axvline(24, color="black", ls="--", alpha=0.5)

ax.set_title("Graf 2BI: 5°C binning feature 2BH")
ax.set_xlabel("Teplota [°C]")
ax.set_ylabel("Feature 2BH [µW·s]")

ax.legend()
ax.grid(True, alpha=0.3)

plt.show()


In [ ]:
# --- Graf 2C: Všechny původní průběhy Channel A y(t) v jednom grafu ---
fig, ax = plt.subplots(figsize=(14, 6.0))
for temp in temps:
    d = datasets[temp]
    ax.plot(d["t"], d["y"] * 1e6, color=cmap(norm(temp)), lw=0.8, alpha=0.7)

ax.set_title("Všechny soubory (0 °C – 48 °C): průběh výkonu Channel A y(t)")
ax.set_xlabel("Čas t [s]")
ax.set_ylabel("Výkon Channel A [µW]")
ax.set_xlim(0, 900)

cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Teplota [°C]")

plt.tight_layout()
plt.show()

In [ ]:
# Načtení všech souborů seřazených numericky podle teploty
files = sorted(DATA_DIR.glob("*.txt"), key=extract_temp)
datasets = {}

for p in files:
    temp = extract_temp(p)
    t, y = load_starlab_file(p)
    dt = np.diff(t)
    dy_abs = np.abs(np.diff(y))          # |y[i+1] - y[i]|  [W]
    dydt_abs = dy_abs / dt               # |dy / dt|        [W/s]
    datasets[temp] = {
        "name": p.name,
        "t": t,
        "y": y,
        "t_diff": t[1:],                 # čas odpovídající rozdílu y[i+1] - y[i]
        "dy_abs": dy_abs,
        "dydt_abs": dydt_abs,
    }

temps = sorted(datasets.keys())
print(f"Načteno {len(datasets)} souborů z: {DATA_DIR.resolve()}")
print(f"Teploty: {temps[0]} °C až {temps[-1]} °C, dt = {np.median(np.diff(datasets[temps[0]]['t'])):.6f} s")

# --- Rozdíl odezvy vůči baseline ---

temps_plot = []
feature_values = []

for temp in temps:
    d = datasets[temp]

    baseline = np.mean(d["y"][d["t"] < 100])

    response = np.mean(
        d["y"][
            (d["t"] >= 200) &
            (d["t"] <= 400)
        ]
    )

    feature = (response - baseline) * 1e6

    temps_plot.append(temp)
    feature_values.append(feature)

plt.figure(figsize=(10, 5))

plt.plot(
    temps_plot,
    feature_values,
    marker="o"
)

plt.axhline(0, color="black", lw=0.8)
plt.axvline(24, color="black", ls="--", alpha=0.5)

plt.xlabel("Teplota [°C]")
plt.ylabel("Δ výkon [µW]")
plt.title("Odezva vůči baseline")

plt.grid(True, alpha=0.3)
plt.show()



In [ ]:
# --- PCA bez sklearn ---

X = []

for temp in temps:
    d = datasets[temp]

    mask = (
        (d["t"] >= 140) &
        (d["t"] <= 400)
    )

    y = d["y"][mask]

    y = y - np.mean(y)

    X.append(y)

X = np.array(X)

# standardizace sloupců
X = X - np.mean(X, axis=0)
X = X / (np.std(X, axis=0) + 1e-12)

# PCA přes SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)

PC1 = U[:, 0] * S[0]
PC2 = U[:, 1] * S[1]

plt.figure(figsize=(8, 6))

sc = plt.scatter(
    PC1,
    PC2,
    c=temps,
    cmap="coolwarm",
    s=60
)

plt.colorbar(sc, label="Teplota [°C]")

for i, temp in enumerate(temps):
    plt.annotate(
        str(temp),
        (PC1[i], PC2[i]),
        fontsize=8
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA průběhů 140–400 s")

plt.grid(True, alpha=0.3)
plt.show()

temps_plot = []
distance_plot = []

ref = datasets[24]["y"]

for temp in temps:
    y = datasets[temp]["y"]

    n = min(len(y), len(ref))

    rms = np.sqrt(
        np.mean(
            (y[:n] - ref[:n])**2
        )
    )

    temps_plot.append(temp)
    distance_plot.append(rms * 1e6)

plt.figure(figsize=(10, 5))

plt.plot(
    temps_plot,
    distance_plot,
    marker="o"
)

plt.axvline(24, color="black", ls="--")

plt.xlabel("Teplota [°C]")
plt.ylabel("RMS odchylka od 24 °C [µW]")
plt.title("Vzdálenost průběhu od referenčního průběhu 24 °C")

plt.grid(True)
plt.show()

# --- PC1 a PC2 jako funkce teploty ---

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(10, 8),
    sharex=True
)

ax1.plot(
    temps,
    PC1,
    marker="o"
)

ax1.axvline(
    24,
    color="black",
    ls="--",
    alpha=0.5
)

ax1.set_ylabel("PC1")
ax1.set_title("První hlavní komponenta")

ax2.plot(
    temps,
    PC2,
    marker="o"
)

ax2.axvline(
    24,
    color="black",
    ls="--",
    alpha=0.5
)

ax2.set_xlabel("Teplota [°C]")
ax2.set_ylabel("PC2")
ax2.set_title("Druhá hlavní komponenta")

ax1.grid(True, alpha=0.3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Graf 2BK2: Časové těžiště odezvy ---

temps_plot = []
tcenter_values = []

T_START = 140
T_STOP = 400

for temp in temps:

    d = datasets[temp]

    baseline = np.mean(d["y"][d["t"] < 100])

    mask = (
        (d["t"] >= T_START)
        & (d["t"] <= T_STOP)
    )

    t = d["t"][mask]
    y = d["y"][mask] - baseline

    w = np.abs(y)

    if np.sum(w) > 0:
        t_center = np.sum(t * w) / np.sum(w)
    else:
        t_center = np.nan

    temps_plot.append(temp)
    tcenter_values.append(t_center)

plt.figure(figsize=(10,5))

plt.plot(
    temps_plot,
    tcenter_values,
    marker="o",
    lw=2
)

plt.axvline(
    24,
    color="black",
    ls="--",
    alpha=0.5
)

plt.xlabel("Teplota [°C]")
plt.ylabel("t_center [s]")
plt.title("2BK2: Časové těžiště odezvy")

plt.grid(True, alpha=0.3)
plt.show()

# --- Graf 2BO: Korelace s průběhem 24 °C ---

T_START = 140
T_STOP = 400

ref_d = datasets[24]

mask_ref = (
    (ref_d["t"] >= T_START) &
    (ref_d["t"] <= T_STOP)
)

ref = ref_d["y"][mask_ref]
ref = ref - np.mean(ref)

temps_plot = []
corr_values = []

for temp in temps:

    d = datasets[temp]

    mask = (
        (d["t"] >= T_START) &
        (d["t"] <= T_STOP)
    )

    y = d["y"][mask]
    y = y - np.mean(y)

    n = min(len(y), len(ref))

    c = np.corrcoef(
        ref[:n],
        y[:n]
    )[0, 1]

    temps_plot.append(temp)
    corr_values.append(c)

plt.figure(figsize=(10,5))

plt.plot(
    temps_plot,
    corr_values,
    marker="o"
)

plt.axvline(24, color="black", ls="--", alpha=0.5)

plt.xlabel("Teplota [°C]")
plt.ylabel("Korelace s 24 °C")
plt.title("2BO: Podobnost tvaru s referenčním průběhem")

plt.grid(True, alpha=0.3)
plt.show()

# --- Graf 2BP: Korelační matice mezi všemi teplotami ---

T_START = 140
T_STOP = 400

signals = []

for temp in temps:

    d = datasets[temp]

    mask = (
        (d["t"] >= T_START) &
        (d["t"] <= T_STOP)
    )

    y = d["y"][mask]
    y = y - np.mean(y)

    signals.append(y)

# sjednocení délky
n = min(len(y) for y in signals)

signals = np.array([
    y[:n]
    for y in signals
])

corr = np.corrcoef(signals)

plt.figure(figsize=(10, 8))

plt.imshow(
    corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    origin="lower"
)

plt.colorbar(label="Korelace")

ticks = np.arange(0, len(temps), 5)

plt.xticks(ticks, [temps[i] for i in ticks], rotation=90)
plt.yticks(ticks, [temps[i] for i in ticks])

plt.xlabel("Teplota [°C]")
plt.ylabel("Teplota [°C]")
plt.title("2BP: Korelační matice průběhů")

plt.tight_layout()
plt.show()


plt.figure(figsize=(8,6))

plt.plot(
    PC1,
    PC2,
    "-o",
    alpha=0.7
)

for i, t in enumerate(temps):
    plt.annotate(str(t), (PC1[i], PC2[i]))

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Trajektorie teplot v PCA prostoru")

plt.grid(True)
plt.show()

In [ ]:
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110

# --------------------------------------------------
# Načtení dat
# --------------------------------------------------

DATA_DIR = Path("../data_correctedTimeStamp")

if not DATA_DIR.exists():
    DATA_DIR = Path("data_correctedTimeStamp")


def extract_temp(path):
    m = re.search(r"(-?\d+)", path.stem)
    return int(m.group(1))


def load_starlab_file(path):
    t_vals = []
    y_vals = []

    in_data = False

    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:

            if not in_data:
                if "Timestamp" in line and "Channel A" in line:
                    in_data = True
                continue

            parts = line.strip().split()

            if len(parts) >= 2:
                t_vals.append(float(parts[0]))
                y_vals.append(float(parts[1]))

    return np.asarray(t_vals), np.asarray(y_vals)


files = sorted(DATA_DIR.glob("*.txt"), key=extract_temp)

datasets = {}

for p in files:

    temp = extract_temp(p)

    t, y = load_starlab_file(p)

    datasets[temp] = {
        "t": t,
        "y": y,
        "name": p.name
    }

temps = sorted(datasets.keys())

print(f"Načteno {len(temps)} teplot.")
print(f"Rozsah: {temps[0]} °C až {temps[-1]} °C")

# --------------------------------------------------
# Seskupení po 5 °C kolem referenčních 24 °C
# --------------------------------------------------

groups = {}

for temp in temps:

    group_temp = 24 + 5 * round((temp - 24) / 5)

    groups.setdefault(group_temp, []).append(temp)

print("\nSkupiny:")

for g in sorted(groups):
    print(f"{g:>3} °C : {groups[g]}")

# --------------------------------------------------
# Průměrné průběhy skupin
# --------------------------------------------------

T_START = 140
T_STOP = 400

group_curves = {}
common_time = None

for group_temp, members in sorted(groups.items()):

    curves = []

    for temp in members:

        d = datasets[temp]

        mask = (
            (d["t"] >= T_START)
            & (d["t"] <= T_STOP)
        )

        t = d["t"][mask]

        # baseline z klidové části
        baseline = np.mean(d["y"][d["t"] < 100])

        y = d["y"][mask] - baseline

        curves.append(y)

        if common_time is None:
            common_time = t

    min_len = min(len(c) for c in curves)

    curves = np.array([
        c[:min_len]
        for c in curves
    ])

    group_curves[group_temp] = np.mean(curves, axis=0)

common_time = common_time[:min_len]

# --------------------------------------------------
# Graf
# --------------------------------------------------

plt.figure(figsize=(12, 6))

for group_temp in sorted(group_curves):

    plt.plot(
        common_time,
        group_curves[group_temp] * 1e6,
        lw=2,
        label=f"{group_temp} °C"
    )

plt.axvline(
    150,
    color="gray",
    ls="--",
    alpha=0.3
)

plt.xlabel("Čas [s]")
plt.ylabel("Δ výkon [µW]")
plt.title("Průměrné průběhy po 5 °C skupinách (centrum 24 °C)")

plt.legend(
    ncol=2,
    fontsize=8
)

plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# ==========================================================
# 2BR - charakteristiky průměrných průběhů po 5 °C skupinách
# ==========================================================

import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# Načtení dat
# --------------------------------------------------

DATA_DIR = Path("../data_correctedTimeStamp")

if not DATA_DIR.exists():
    DATA_DIR = Path("data_correctedTimeStamp")


def extract_temp(path):
    m = re.search(r"(-?\d+)", path.stem)
    return int(m.group(1))


def load_starlab_file(path):
    t_vals = []
    y_vals = []

    in_data = False

    with open(path, encoding="utf-8", errors="replace") as f:

        for line in f:

            if not in_data:

                if "Timestamp" in line and "Channel A" in line:
                    in_data = True

                continue

            parts = line.strip().split()

            if len(parts) >= 2:
                t_vals.append(float(parts[0]))
                y_vals.append(float(parts[1]))

    return np.asarray(t_vals), np.asarray(y_vals)


# --------------------------------------------------
# Načtení všech souborů
# --------------------------------------------------

files = sorted(DATA_DIR.glob("*.txt"), key=extract_temp)

datasets = {}

for p in files:

    temp = extract_temp(p)

    t, y = load_starlab_file(p)

    datasets[temp] = {
        "t": t,
        "y": y,
    }

temps = sorted(datasets.keys())

# --------------------------------------------------
# Seskupení po 5 °C kolem 24 °C
# --------------------------------------------------

groups = {}

for temp in temps:

    group_temp = 24 + 5 * round((temp - 24) / 5)

    groups.setdefault(group_temp, []).append(temp)

# --------------------------------------------------
# Průměrné průběhy
# --------------------------------------------------

T_START = 140
T_STOP = 400

features = []

for group_temp, members in sorted(groups.items()):

    curves = []
    common_t = None

    for temp in members:

        d = datasets[temp]

        mask = (
            (d["t"] >= T_START)
            & (d["t"] <= T_STOP)
        )

        t = d["t"][mask]

        baseline = np.mean(
            d["y"][d["t"] < 100]
        )

        y = d["y"][mask] - baseline

        if common_t is None:
            common_t = t

        curves.append(y)

    min_len = min(len(c) for c in curves)

    Y = np.vstack([
        c[:min_len]
        for c in curves
    ])

    t = common_t[:min_len]

    mean_curve = np.mean(Y, axis=0)

    idx_max = np.argmax(mean_curve)
    idx_min = np.argmin(mean_curve)

    y_max = mean_curve[idx_max] * 1e6
    y_min = mean_curve[idx_min] * 1e6

    features.append({
        "group": group_temp,
        "ymax": y_max,
        "ymin": y_min,
        "amp": y_max - y_min,
        "tmax": t[idx_max],
        "tmin": t[idx_min],
    })

# --------------------------------------------------
# Pole pro grafy
# --------------------------------------------------

groups_x = [f["group"] for f in features]

ymax = [f["ymax"] for f in features]
ymin = [f["ymin"] for f in features]
amp  = [f["amp"]  for f in features]

tmax = [f["tmax"] for f in features]
tmin = [f["tmin"] for f in features]

# --------------------------------------------------
# Grafy
# --------------------------------------------------

fig, axs = plt.subplots(
    2, 2,
    figsize=(13, 8),
    constrained_layout=True
)

# ymax

axs[0,0].plot(groups_x, ymax, "o-")
axs[0,0].axvline(24, ls="--", color="black", alpha=0.5)
axs[0,0].set_title("Maximum odezvy")
axs[0,0].set_ylabel("y_max [µW]")

# ymin

axs[0,1].plot(groups_x, ymin, "o-", color="tab:red")
axs[0,1].axvline(24, ls="--", color="black", alpha=0.5)
axs[0,1].set_title("Minimum odezvy")
axs[0,1].set_ylabel("y_min [µW]")

# amplituda

axs[1,0].plot(groups_x, amp, "o-", color="tab:green")
axs[1,0].axvline(24, ls="--", color="black", alpha=0.5)
axs[1,0].set_title("Amplituda")
axs[1,0].set_ylabel("y_max - y_min [µW]")
axs[1,0].set_xlabel("Teplota skupiny [°C]")

# časování

axs[1,1].plot(groups_x, tmax, "o-", label="t_max")
axs[1,1].plot(groups_x, tmin, "s-", label="t_min")

axs[1,1].axvline(24, ls="--", color="black", alpha=0.5)

axs[1,1].set_title("Časy extrémů")
axs[1,1].set_ylabel("čas [s]")
axs[1,1].set_xlabel("Teplota skupiny [°C]")
axs[1,1].legend()

for ax in axs.ravel():
    ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
# --- 2BS: Teplotní podpis (ymax, ymin) ---

plt.figure(figsize=(8, 8))

for f in features:

    plt.scatter(
        f["ymax"],
        f["ymin"],
        s=100
    )

    plt.annotate(
        f'{f["group"]}°C',
        (f["ymax"], f["ymin"])
    )

plt.xlabel("y_max [µW]")
plt.ylabel("y_min [µW]")
plt.title("2BS: Teplotní podpis skupin")

plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# --- 2BT: amplituda vs asymetrie ---

plt.figure(figsize=(8,8))

for f in features:

    amp = f["ymax"] - f["ymin"]
    asym = f["ymax"] + f["ymin"]

    plt.scatter(
        amp,
        asym,
        s=120
    )

    plt.annotate(
        f'{f["group"]}°C',
        (amp, asym)
    )

plt.xlabel("Amplituda [µW]")
plt.ylabel("Asymetrie [µW]")
plt.title("2BT: Amplituda vs asymetrie")

plt.grid(True, alpha=0.3)

plt.show()


In [ ]:
# ===========================================
# 2BU - 3D podpis teplotních tříd
# ===========================================

import numpy as np
import pandas as pd

rows = []

for f in features:

    amp = f["ymax"] - f["ymin"]
    asym = f["ymax"] + f["ymin"]

    rows.append({
        "temp": f["group"],
        "amp": amp,
        "asym": asym,
        "tmax": f["tmax"],
        "tmin": f["tmin"]
    })

df = pd.DataFrame(rows)

print(df.round(2))

# párové vzdálenosti mezi třídami

X = df[["amp", "asym"]].values

N = len(X)

dist = np.zeros((N, N))

for i in range(N):
    for j in range(N):

        dist[i, j] = np.linalg.norm(
            X[i] - X[j]
        )

plt.figure(figsize=(8, 7))

plt.imshow(
    dist,
    cmap="viridis"
)

plt.colorbar(label="Vzdálenost")

ticks = np.arange(len(df))

plt.xticks(ticks, df["temp"], rotation=90)
plt.yticks(ticks, df["temp"])

plt.title("2BU: Vzdálenosti mezi 5°C třídami")

plt.xlabel("Teplota [°C]")
plt.ylabel("Teplota [°C]")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 2BV - klasifikace podle nejbližšího podpisu
# ==========================================

import numpy as np
import pandas as pd

rows = []

for f in features:

    rows.append([
        f["group"],
        f["ymax"],
        f["ymin"]
    ])

df = pd.DataFrame(
    rows,
    columns=["temp", "ymax", "ymin"]
)

X = df[["ymax", "ymin"]].values

temps_cls = df["temp"].values

print("Nejbližší skupina:")

for i in range(len(X)):

    dists = []

    for j in range(len(X)):

        if i == j:
            continue

        d = np.linalg.norm(
            X[i] - X[j]
        )

        dists.append((d, temps_cls[j]))

    dists.sort()

    nearest = dists[0][1]

    print(
        f"{temps_cls[i]:>3} °C  →  "
        f"nejbližší je {nearest:>3} °C"
    )


In [ ]:
# --- 2BW: amplituda proti vzdálenosti od 24 °C ---

deltaT = []
amp = []

for f in features:

    deltaT.append(abs(f["group"] - 24))

    amp.append(
        f["ymax"] - f["ymin"]
    )

plt.figure(figsize=(8,5))

plt.scatter(
    deltaT,
    amp,
    s=120
)

for f in features:

    plt.annotate(
        f'{f["group"]}°C',
        (
            abs(f["group"] - 24),
            f["ymax"] - f["ymin"]
        )
    )

plt.xlabel("|T - 24| [°C]")
plt.ylabel("Amplituda [µW]")

plt.title(
    "2BW: Odezva vs vzdálenost od referenční teploty"
)

plt.grid(True)

plt.show()